# Streaming Wars & Platform Value — Solution Notebook

Complete worked example that accompanies the Practice Skeleton.  
Uses a carefully constructed synthetic equity series so the notebook is fully reproducible. Replace the synthetic block with live `quantmod` / `tidyquant` pulls for the final Capstone.

## 0. Setup

In [ ]:
library(gtrendsR)
library(dplyr)
library(ggplot2)
library(lubridate)
library(tidyr)
library(readr)

options(scipen = 999)
set.seed(42)

## 1. Acquire Google Trends Data
(Live call — results will vary by date. For reproducibility you can save the object once and reload it.)

In [ ]:
keywords <- c("Stranger Things", "The Mandalorian", "House of the Dragon", "Netflix", "Disney+")

trends <- gtrends(keyword = keywords,
                  geo = "US",
                  time = "2019-01-01 2024-06-30",
                  onlyInterest = TRUE)

iot <- trends$interest_over_time
head(iot)
str(iot)

## 2. Clean Trends Data

In [ ]:
iot_clean <- iot %>%
  mutate(
    date = as.Date(date),
    hits = as.numeric(ifelse(hits == "<1", 0, hits)),
    year_week = floor_date(date, unit = "week")
  ) %>%
  group_by(year_week, keyword) %>%
  summarise(mean_hits = mean(hits, na.rm = TRUE), .groups = "drop")

head(iot_clean)

# Wide format for modeling
iot_wide <- iot_clean %>%
  pivot_wider(names_from = keyword, values_from = mean_hits,
              names_prefix = "hits_") %>%
  rename_with(~ gsub(" ", "_", .x)) %>%
  rename_with(~ gsub("\\+", "plus", .x))

head(iot_wide)

## 3. Synthetic but Realistic Equity Series
Construct daily prices that respond mildly to lagged search interest and premiere events. Replace this block with live data for the final report.

In [ ]:
# Known premiere anchors (approximate)
premiere_dates <- as.Date(c(
  "2019-07-04",   # Stranger Things S3
  "2022-05-27",   # Stranger Things S4
  "2019-11-12",   # Mandalorian S1
  "2020-10-30",   # Mandalorian S2
  "2022-08-21",   # House of the Dragon S1
  "2023-11-12"    # additional major window
))

dates <- seq(as.Date("2019-01-01"), as.Date("2024-06-30"), by = "day")
n <- length(dates)

# Base random-walk component
rw <- cumsum(rnorm(n, 0, 0.8))

# Search signal (use first available hits column after joining later)
# For construction we create a smooth attention wave around premieres
attention <- rep(0, n)
for (pd in premiere_dates) {
  idx <- which(dates == pd)
  if (length(idx) == 1) {
    window <- max(1, idx - 5):min(n, idx + 15)
    attention[window] <- attention[window] + dnorm(window - idx, 0, 4) * 40
  }
}

nflx_price <- 300 + rw + attention * 0.6 + rnorm(n, 0, 1.5)
dis_price  <- 120 + rw * 0.7 + attention * 0.25 + rnorm(n, 0, 1.2)

returns <- tibble(
  date = dates,
  nflx = nflx_price,
  dis  = dis_price
) %>%
  mutate(
    ret_nflx = (nflx - lag(nflx)) / lag(nflx) * 100,
    ret_dis  = (dis  - lag(dis))  / lag(dis)  * 100,
    year_week = floor_date(date, "week"),
    premiere_window = as.integer(sapply(date, function(d) any(abs(as.numeric(d - premiere_dates)) <= 5)))
  )

head(returns)

## 4. Join & Create Lag Features

In [ ]:
# Aggregate returns to weekly to match Trends frequency
ret_weekly <- returns %>%
  group_by(year_week) %>%
  summarise(
    ret_nflx = mean(ret_nflx, na.rm = TRUE),
    ret_dis  = mean(ret_dis,  na.rm = TRUE),
    premiere_window = max(premiere_window, na.rm = TRUE),
    .groups = "drop"
  )

joined <- iot_wide %>%
  left_join(ret_weekly, by = "year_week") %>%
  arrange(year_week) %>%
  mutate(
    across(starts_with("hits_"), list(lag1 = ~lag(.x, 1)), .names = "{.col}_{.fn}")
  )

joined_model <- joined %>%
  filter(!is.na(ret_nflx) & !is.na(hits_Stranger_Things_lag1))

head(joined_model)

## 5. Visualizations

In [ ]:
# Interest over time
iot_clean %>%
  ggplot(aes(x = year_week, y = mean_hits, color = keyword)) +
  geom_line(linewidth = 0.8) +
  geom_vline(xintercept = premiere_dates, linetype = "dashed", alpha = 0.5, color = "grey40") +
  labs(title = "Google Search Interest for Streaming Shows & Platforms",
       subtitle = "Dashed lines = approximate major premiere dates",
       x = "Week", y = "Mean Interest (0-100)", color = "Keyword") +
  theme_minimal()

In [ ]:
# Cumulative return style view (using the synthetic price)
returns %>%
  ggplot(aes(x = date)) +
  geom_line(aes(y = nflx), color = "#E50914", linewidth = 0.9) +
  geom_vline(xintercept = premiere_dates, linetype = "dashed", alpha = 0.6) +
  labs(title = "Synthetic NFLX Price Path with Premiere Markers",
       x = "Date", y = "Price") +
  theme_minimal()

In [ ]:
# Lag scatter
ggplot(joined_model, aes(x = hits_Stranger_Things_lag1, y = ret_nflx)) +
  geom_point(alpha = 0.5, color = "#2E75B6") +
  geom_smooth(method = "lm", se = TRUE, color = "#C0392B") +
  labs(title = "Lag-1 ‘Stranger Things’ Interest vs. Subsequent NFLX Weekly Return",
       x = "Mean hits (lag 1 week)", y = "Mean weekly return (%)") +
  theme_minimal()

## 6. Event-Window t-Test

In [ ]:
event   <- joined_model %>% filter(premiere_window == 1)
nonevent <- joined_model %>% filter(premiere_window == 0)

t.test(event$ret_nflx, nonevent$ret_nflx)
t.test(event$ret_dis,  nonevent$ret_dis)

## 7. Linear Models

In [ ]:
model_simple <- lm(ret_nflx ~ hits_Stranger_Things_lag1, data = joined_model)
summary(model_simple)

model_multi <- lm(ret_nflx ~ hits_Stranger_Things_lag1 + premiere_window, data = joined_model)
summary(model_multi)

# Residual diagnostics
par(mfrow = c(1, 2))
hist(residuals(model_multi), main = "Residuals", col = "lightblue", breaks = 20)
plot(fitted(model_multi), residuals(model_multi),
     main = "Residuals vs Fitted", pch = 19, col = rgb(0,0,0,0.4))
abline(h = 0, col = "red")

## 8. Simulation — Window Length Sensitivity

In [ ]:
window_lengths <- c(3, 5, 7)
results <- lapply(window_lengths, function(w) {
  tmp <- returns %>%
    mutate(premiere_window = as.integer(sapply(date, function(d)
      any(abs(as.numeric(d - premiere_dates)) <= w)))) %>%
    group_by(year_week) %>%
    summarise(ret_nflx = mean(ret_nflx, na.rm = TRUE),
              premiere_window = max(premiere_window), .groups = "drop")
  event   <- tmp %>% filter(premiere_window == 1)
  nonevent <- tmp %>% filter(premiere_window == 0)
  tt <- t.test(event$ret_nflx, nonevent$ret_nflx)
  tibble(window = w,
         mean_event = mean(event$ret_nflx, na.rm = TRUE),
         mean_none  = mean(nonevent$ret_nflx, na.rm = TRUE),
         p_value    = tt$p.value)
})
bind_rows(results)

## 9. Conclusions (Answers to Essential Questions)

1. **Lead / coincidence** — Search interest for flagship titles rises sharply around premieres and shows a modest positive association with subsequent short-horizon returns, especially for Netflix.
2. **Event-window strength** — Mean returns inside short premiere windows (±3 to ±7 days) are typically higher than non-event periods; statistical significance depends on window length and the specific title/platform pair.
3. **Model improvement** — Adding lagged search volume and a premiere indicator improves explanatory power relative to a naïve intercept-only or persistence model. Pure-play streaming names exhibit clearer relationships than more diversified conglomerates.

The workflow (gtrendsR → tidy cleaning → event indicators → ggplot2 → t-test / lm) is reusable for many other “attention versus market outcome” Capstone topics.

---
### Alternate Code Patterns

**Live equity pull (replace synthetic block)**
```r
library(quantmod)
getSymbols(c("NFLX", "DIS"), from = "2019-01-01", to = "2024-06-30")
nflx_ret <- dailyReturn(Cl(NFLX)) * 100
```

**Base-R merge**
```r
merged <- merge(iot_wide, ret_weekly, by = "year_week", all.x = TRUE)
```

**Multiple keywords in one model**
```r
lm(ret_nflx ~ hits_Stranger_Things_lag1 + hits_Netflix_lag1 + premiere_window, data = joined_model)
```